# Подбор гиперпараметров для HistGradientBoostingClassifier с помощью Optuna

1.  Определяем, какие из 41 банковского продукта являются «ультраредкими» (ultra_rare), а какие — «нормальными» (normal_targets) на основе их распространённости среди клиентов.

Фильтрация по порогу 0.005 (0.5%):

- ultra_rare — продукты, доля владения которыми строго меньше 0.5%. Такие целевые переменные крайне несбалансированы (положительных примеров очень мало).

- normal_targets — продукты с долей ≥ 0.5%.

2. Выполняется цикл по всем целевым переменным (target_columns): каждый продукт — отдельная бинарная задача.

2.1. В каждом шаге цикла создается подвыборка размером 10000 строк:

- для «ультраредких» продуктов (ultra_rare) с помощью адаптивного семплинга *get_subsample*, 

- для обычных продуктов — train_test_split с train_size=10000 и стратификацией, чтобы сохранить долю положительных примеров.

Функция  *get_subsample* для  для «ультраредких» продуктов создаёт сбалансированную (или целенаправленно сформированную) подвыборку фиксированного размера n_total (по умолчанию 10 000) для обучения модели при подборе гиперпараметров:

a. Находятся индексы положительных (y==1) и отрицательных (y==0) объектов.

b. Случай 1: положительных объектов больше, чем нужно (n_pos > n_total):

берётся ровно n_total // 2 положительных (половина выборки) и n_total - n_pos_sample отрицательных (вторая половина),

в итоге получается строго сбалансированная подвыборка 50/50.

c. Случай 2: положительных меньше или равно n_total:

берутся все положительные объекты, недостающее до n_total количество добирается отрицательными.

Если отрицательных не хватает (крайний случай), берутся все доступные отрицательные.

*Пояснение:* для ultra_rare используется get_subsample, а не train_test_split со стратификацией, потому что при доле <0.5% в случайной подвыборке на 10 000 может не оказаться ни одного положительного примера, и стратификации может оказаться недостаточно (если общее число положительных < 10 000, стратификация всё равно даст очень мало позитивных объектов).


2.2. Для каждой подвыборки запускается Optuna Study с целью максимизировать средний ROC-AUC, полученный через 3-кратную кросс-валидацию. Функция objective предлагает гиперпараметры (max_iter, max_depth, learning_rate и др.), обучает HistGradientBoostingClassifier и возвращает среднее значение метрики. Поиск выполняется за 20 итераций (n_trials=20) параллельно (n_jobs=-1). Параметр gc_after_trial=True освобождает память после каждой попытки. Лучшие параметры сохраняются в best_params_dict.

2.3. Сохранение словаря гиперпараметров для каждого продукта в файл *best_params_dict.pkl*.

In [1]:
!pip install polars

In [2]:
import gc
import numpy as np
import pandas as pd
import polars as pl
import optuna
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import pickle

In [3]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [4]:
target = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_target.parquet')

In [5]:
target_columns = [col for col in target.columns if col.startswith("target")]

In [6]:
target_col_means = (
    target.select(pl.col(target_columns).mean())
    .transpose(include_header = True, column_names = ["Mean"])
    .rename({"column": "Target"})
)
print(target_col_means)

shape: (41, 2)
┌─────────────┬──────────┐
│ Target      ┆ Mean     │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ target_1_1  ┆ 0.010396 │
│ target_1_2  ┆ 0.003425 │
│ target_1_3  ┆ 0.023785 │
│ target_1_4  ┆ 0.023429 │
│ target_1_5  ┆ 0.001839 │
│ …           ┆ …        │
│ target_9_5  ┆ 0.006583 │
│ target_9_6  ┆ 0.223072 │
│ target_9_7  ┆ 0.077248 │
│ target_9_8  ┆ 0.010433 │
│ target_10_1 ┆ 0.315052 │
└─────────────┴──────────┘


In [7]:
ultra_rare = (
    target_col_means
    .filter(pl.col("Mean") < 0.005)
    .select("Target")
    .to_series()
    .to_list()
)
normal_targets = (
    target_col_means
    .filter(pl.col("Mean") >= 0.005)
    .select("Target")
    .to_series()
    .to_list()
)

In [8]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1300/train_combined_1300.parquet')

print('Тренировочные данные:', train.shape)

Тренировочные данные: (750000, 1301)


In [9]:
X_train_final = train.drop('customer_id')

In [10]:
X_np = X_train_final.to_numpy().astype('float32')

In [11]:
y_np = (
    target
    .drop('customer_id')
    .to_numpy()
    .astype('int8')
)

In [12]:
del X_train_final
del target

gc.collect()

0

In [13]:
def get_subsample(X, y, n_total=10000, random_state=42):
    np.random.seed(random_state)
    # Индексы положительных и отрицательных классов
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos > n_total:
        # Если положительных больше, чем нужно, берём половину от n_total (или пропорционально)
        n_pos_sample = n_total // 2
        n_neg_sample = n_total - n_pos_sample
        pos_sample_idx = np.random.choice(pos_idx, size=n_pos_sample, replace=False)
        neg_sample_idx = np.random.choice(neg_idx, size=n_neg_sample, replace=False)
    else:
        # Берём все положительные, остальное добираем отрицательными
        n_neg_needed = n_total - n_pos
        if n_neg_needed > n_neg:
            neg_sample_idx = neg_idx  # если отрицательных не хватает, берём все
        else:
            neg_sample_idx = np.random.choice(neg_idx, size=n_neg_needed, replace=False)
        pos_sample_idx = pos_idx

    idx = np.concatenate([pos_sample_idx, neg_sample_idx])
    # Возвращаем подвыборки
    X_sub = X[idx]
    y_sub = y[idx]
    return X_sub, y_sub

In [14]:
def objective(trial, X, y):
    params = {
        'max_iter': trial.suggest_int('max_iter', 200, 600),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 50),
        'l2_regularization': trial.suggest_float('l2_regularization', 0.0, 5.0),
        'max_bins': trial.suggest_int('max_bins', 32, 128),
        'validation_fraction': 0.1,
        'early_stopping': True,
        'n_iter_no_change': trial.suggest_int('n_iter_no_change', 5, 20),
        'random_state': 42
    }
    model = HistGradientBoostingClassifier(**params)
    
    scores = cross_val_score(model, X, y, cv=3, scoring='roc_auc')
    return scores.mean()

In [15]:
best_params_dict = {}

In [16]:
# Для каждого таргета отдельно
for i, t in enumerate(target_columns):
    print(f"Optimizing for {t}")
    y_one = y_np[:, i]  

    if t in ultra_rare:
        X_sub, y_sub = get_subsample(X_np, y_one, n_total=10000)
    else:
        from sklearn.model_selection import train_test_split
        X_sub, _, y_sub, _ = train_test_split(X_np, y_one, train_size=10000,
                                              stratify=y_one, random_state=42)

    study = optuna.create_study(direction='maximize', study_name=f'hgb_opt_{t}')
    study.optimize(lambda trial: objective(trial, X_sub, y_sub), n_trials=20, n_jobs=-1, gc_after_trial=True)

    best_params = study.best_params
    print(f"Best params for {t}: {best_params}")
  
    best_params_dict[t] = study.best_params
    
    del study
    del X_sub
    del y_sub
    gc.collect()

Optimizing for target_1_1


[I 2026-04-06 07:53:26,452] A new study created in memory with name: hgb_opt_target_1_1
[I 2026-04-06 07:53:51,329] Trial 2 finished with value: 0.823065206995477 and parameters: {'max_iter': 400, 'max_depth': 4, 'learning_rate': 0.09952762082646548, 'min_samples_leaf': 17, 'l2_regularization': 4.681557558116698, 'max_bins': 88, 'n_iter_no_change': 5}. Best is trial 2 with value: 0.823065206995477.
[I 2026-04-06 07:53:53,259] Trial 0 finished with value: 0.8271499336696521 and parameters: {'max_iter': 354, 'max_depth': 9, 'learning_rate': 0.14777910406227623, 'min_samples_leaf': 35, 'l2_regularization': 0.7571010044462984, 'max_bins': 40, 'n_iter_no_change': 15}. Best is trial 0 with value: 0.8271499336696521.
[I 2026-04-06 07:53:53,750] Trial 3 finished with value: 0.8279691009724516 and parameters: {'max_iter': 332, 'max_depth': 4, 'learning_rate': 0.09953265905942386, 'min_samples_leaf': 23, 'l2_regularization': 2.212573093237035, 'max_bins': 79, 'n_iter_no_change': 14}. Best is tri

Best params for target_1_1: {'max_iter': 483, 'max_depth': 6, 'learning_rate': 0.04595171487921697, 'min_samples_leaf': 50, 'l2_regularization': 2.7962024337395355, 'max_bins': 127, 'n_iter_no_change': 17}
Optimizing for target_1_2


[I 2026-04-06 07:56:49,735] A new study created in memory with name: hgb_opt_target_1_2
[I 2026-04-06 07:57:58,129] Trial 2 finished with value: 0.8311434430338543 and parameters: {'max_iter': 522, 'max_depth': 10, 'learning_rate': 0.0530519363681705, 'min_samples_leaf': 37, 'l2_regularization': 0.5290992471538303, 'max_bins': 93, 'n_iter_no_change': 5}. Best is trial 2 with value: 0.8311434430338543.
[I 2026-04-06 07:57:59,453] Trial 1 finished with value: 0.8291335940110846 and parameters: {'max_iter': 434, 'max_depth': 10, 'learning_rate': 0.06766222722632767, 'min_samples_leaf': 42, 'l2_regularization': 0.19024050559050154, 'max_bins': 111, 'n_iter_no_change': 11}. Best is trial 2 with value: 0.8311434430338543.
[I 2026-04-06 07:58:30,282] Trial 5 finished with value: 0.8205450622394258 and parameters: {'max_iter': 336, 'max_depth': 8, 'learning_rate': 0.19818117774582214, 'min_samples_leaf': 14, 'l2_regularization': 0.18756114941206325, 'max_bins': 32, 'n_iter_no_change': 18}. Bes

Best params for target_1_2: {'max_iter': 587, 'max_depth': 8, 'learning_rate': 0.014539335286522492, 'min_samples_leaf': 47, 'l2_regularization': 2.9664026486819157, 'max_bins': 61, 'n_iter_no_change': 14}
Optimizing for target_1_3


...

Best params for target_9_7: {'max_iter': 235, 'max_depth': 4, 'learning_rate': 0.01103358511224898, 'min_samples_leaf': 12, 'l2_regularization': 2.927543924518141, 'max_bins': 43, 'n_iter_no_change': 19}
Optimizing for target_9_8


[I 2026-04-06 11:29:35,964] A new study created in memory with name: hgb_opt_target_9_8
[I 2026-04-06 11:29:59,123] Trial 3 finished with value: 0.8789616100500318 and parameters: {'max_iter': 277, 'max_depth': 7, 'learning_rate': 0.18584268688232713, 'min_samples_leaf': 46, 'l2_regularization': 4.604050375751545, 'max_bins': 41, 'n_iter_no_change': 14}. Best is trial 3 with value: 0.8789616100500318.
[I 2026-04-06 11:30:02,005] Trial 0 finished with value: 0.8817386463402466 and parameters: {'max_iter': 376, 'max_depth': 7, 'learning_rate': 0.08562723337988945, 'min_samples_leaf': 45, 'l2_regularization': 3.346426966396874, 'max_bins': 68, 'n_iter_no_change': 8}. Best is trial 0 with value: 0.8817386463402466.
[I 2026-04-06 11:30:05,109] Trial 2 finished with value: 0.8675072763454748 and parameters: {'max_iter': 579, 'max_depth': 10, 'learning_rate': 0.06180290087050855, 'min_samples_leaf': 19, 'l2_regularization': 3.1603985027322623, 'max_bins': 77, 'n_iter_no_change': 5}. Best is t

Best params for target_9_8: {'max_iter': 497, 'max_depth': 6, 'learning_rate': 0.04199602255774254, 'min_samples_leaf': 8, 'l2_regularization': 2.083607729226604, 'max_bins': 124, 'n_iter_no_change': 20}
Optimizing for target_10_1


[I 2026-04-06 11:33:02,388] A new study created in memory with name: hgb_opt_target_10_1
[I 2026-04-06 11:33:33,306] Trial 2 finished with value: 0.735119313736812 and parameters: {'max_iter': 417, 'max_depth': 5, 'learning_rate': 0.11457524367154559, 'min_samples_leaf': 11, 'l2_regularization': 0.24566751392118558, 'max_bins': 124, 'n_iter_no_change': 6}. Best is trial 2 with value: 0.735119313736812.
[I 2026-04-06 11:33:35,440] Trial 0 finished with value: 0.729766843660201 and parameters: {'max_iter': 568, 'max_depth': 10, 'learning_rate': 0.1170245362081936, 'min_samples_leaf': 43, 'l2_regularization': 4.624512035802123, 'max_bins': 44, 'n_iter_no_change': 8}. Best is trial 2 with value: 0.735119313736812.
[I 2026-04-06 11:34:01,968] Trial 4 finished with value: 0.7261588609832882 and parameters: {'max_iter': 596, 'max_depth': 7, 'learning_rate': 0.2526478259811527, 'min_samples_leaf': 24, 'l2_regularization': 4.9713438310699, 'max_bins': 116, 'n_iter_no_change': 12}. Best is trial

Best params for target_10_1: {'max_iter': 300, 'max_depth': 4, 'learning_rate': 0.010107202629976548, 'min_samples_leaf': 50, 'l2_regularization': 3.466726427896077, 'max_bins': 86, 'n_iter_no_change': 20}


In [17]:
with open('best_params_dict.pkl', 'wb') as f:
    pickle.dump(best_params_dict, f)